# Sliding window: turning a time series into input–output pairs

A window of length **L** slides along the series one step at a time. Each window
becomes an **input** (blue) and the value right after it becomes the **output**
target (orange). A series of length N gives **N − L** pairs.

**Interactive** (sliders + play button) — needs `ipywidgets`

In [1]:
# Run once if these aren't installed, then restart the kernel
# %pip install numpy matplotlib ipywidgets

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#b4b2a9"

# Example series (replace with your own 1-D array)
N = 24
t = np.arange(N)
data = np.round(10 + 4 * np.sin(t / 2.6) + 0.12 * t + 1.1 * np.sin(t * 1.7 + 1), 1)


def make_windows(series, L):
    """Return X with shape (N-L, L) and y with shape (N-L,)."""
    series = np.asarray(series)
    X = np.array([series[i : i + L] for i in range(len(series) - L)])
    y = np.array([series[i + L] for i in range(len(series) - L)])
    return X, y


X, y = make_windows(data, L=5)
print("X shape:", X.shape, " y shape:", y.shape)

X shape: (19, 5)  y shape: (19,)


In [3]:
def draw_window(i=0, L=5, series=data, max_rows=6):
    """Plot the series with window i highlighted, plus the X / y table below."""
    series = np.asarray(series)
    n = len(series)
    M = n - L  # number of pairs
    i = min(i, M - 1)
    tt = np.arange(1, n + 1)  # 1-based time steps, like the diagram

    fig, (ax, axm) = plt.subplots(
        2, 1, figsize=(11, 7), gridspec_kw={"height_ratios": [1.3, 1]}
    )

    # --- top: time series ---------------------------------------------------
    ax.axvspan(i + 0.5, i + L + 0.5, color=BLUE, alpha=0.12, label="Input window")
    ax.axvspan(
        i + L + 0.5, i + L + 1.5, color=ORANGE, alpha=0.18, label="Output target"
    )
    ax.plot(tt, series, color=GRAY, lw=2, zorder=1)
    ax.plot(tt[i : i + L], series[i : i + L], color=BLUE, lw=2.5, zorder=2)
    ax.plot(
        tt[i + L - 1 : i + L + 1],
        series[i + L - 1 : i + L + 1],
        color=ORANGE,
        lw=2,
        ls="--",
        zorder=2,
    )

    colors = [
        BLUE if i <= k < i + L else ORANGE if k == i + L else GRAY for k in range(n)
    ]
    sizes = [90 if k == i + L else 45 for k in range(n)]
    ax.scatter(
        tt, series, c=colors, s=sizes, edgecolor="white", linewidth=1.5, zorder=3
    )

    ax.set_xlim(0.3, n + 0.7)
    ax.set_xlabel("time step t")
    ax.set_title(
        f"Pair {i + 1} of {M}:  input = t {i + 1}…{i + L},  output = t {i + L + 1}"
        f"   (N={n}, L={L} → N−L = {M} pairs)",
        fontsize=11,
    )
    ax.grid(axis="y", color="#e1e0d9")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper left", frameon=False)

    # --- bottom: X matrix and y vector ---------------------------------------
    axm.axis("off")
    start = max(0, min(i - 2, M - max_rows))
    rows = range(start, min(start + max_rows, M))
    cw, rh, gap = 1.0, 1.0, 1.2
    axm.text(0, 0.6, f"X  (shape {M} × {L})", fontsize=11, va="bottom")
    axm.text(L * cw + gap, 0.6, f"y  (shape {M},)", fontsize=11, va="bottom")
    for r_idx, r in enumerate(rows):
        yy = -(r_idx + 1) * rh
        on = r == i
        axm.text(
            -0.2,
            yy + 0.45,
            f"#{r + 1}",
            ha="right",
            va="center",
            fontsize=9,
            color="black" if on else "gray",
        )
        for c in range(L):
            axm.add_patch(
                Rectangle(
                    (c * cw, yy),
                    cw * 0.95,
                    rh * 0.85,
                    facecolor=BLUE if on else "none",
                    alpha=0.15 if on else 1,
                    edgecolor=BLUE if on else "#d3d1c7",
                )
            )
            axm.text(
                c * cw + cw * 0.475,
                yy + 0.42,
                f"{series[r + c]}",
                ha="center",
                va="center",
                fontsize=9,
                family="monospace",
                color="black" if on else "gray",
            )
        axm.add_patch(
            Rectangle(
                (L * cw + gap, yy),
                cw * 1.1,
                rh * 0.85,
                facecolor=ORANGE if on else "none",
                alpha=0.2 if on else 1,
                edgecolor=ORANGE if on else "#d3d1c7",
            )
        )
        axm.text(
            L * cw + gap + 0.55,
            yy + 0.42,
            f"{series[r + L]}",
            ha="center",
            va="center",
            fontsize=9,
            family="monospace",
            color="black" if on else "gray",
        )
        if on:
            axm.annotate(
                "",
                xy=(L * cw + gap - 0.05, yy + 0.42),
                xytext=(L * cw + 0.05, yy + 0.42),
                arrowprops={"arrowstyle": "->", "color": "black"},
            )
    axm.set_xlim(-1, L * cw + gap + 4)
    axm.set_ylim(-(max_rows + 0.5) * rh, 1.2)
    fig.tight_layout()
    return fig


# draw_window(i=0, L=5); plt.show()

## Interactive version (sliders + play)

Drag **i** to slide the window, change **L** to see how the number of pairs changes,
or press ▶ to animate. Works in Jupyter Notebook, JupyterLab, VS Code and Google Colab.

In [4]:
import ipywidgets as widgets
from IPython.display import display

L_slider = widgets.IntSlider(value=5, min=2, max=10, description="Window L")
i_slider = widgets.IntSlider(value=0, min=0, max=N - 5 - 1, description="Position i")
play = widgets.Play(value=0, min=0, max=N - 5 - 1, interval=700, description="Play")
widgets.jslink((play, "value"), (i_slider, "value"))


def _update_range(change):
    m = N - change["new"] - 1
    i_slider.max = play.max = m
    i_slider.value = min(i_slider.value, m)


L_slider.observe(_update_range, names="value")

out = widgets.interactive_output(
    lambda i, L: (draw_window(i, L), plt.show()), {"i": i_slider, "L": L_slider}
)
display(widgets.VBox([widgets.HBox([play, i_slider, L_slider]), out]))